# nb_01_metadata_delta — scan the source, detect changes, maintain `file_metadata`

**Pipeline 1.** Lists the configured source, computes a change hash, and MERGEs into
`file_metadata`, driving the `process_status` state machine. Also detects deletions via a stale
`last_seen_utc`.

**Source is config-switchable (`source_mode`):**
- `s3_shortcut` — recursively lists the OneLake **S3 shortcut** (Spark-native `binaryFile`, no
  driver-side walk).
- `s3_direct` — lists **directly from S3** over the REST API with AWS SigV4 (no boto3, no
  `%pip`). See `scripts/s3_rest.py` for the reference implementation.

Either way the file identity is the **src_key** (object path relative to the bucket root, e.g.
`testset/finance/reports/e2e/x.pdf`) so change detection and ACLs are mode-independent.

> **Do not run this concurrently with `nb_03`.** Both write the same `file_metadata` Delta table;
> run them in sequence (`nb_01` → `nb_03`). See PRODUCT_SPEC.md §8.

See `PRODUCT_SPEC.md` sections 5, 7.1, 9, 12.


## Parameters + config


In [ ]:
from datetime import datetime, timezone

cfg = {r['key']: r['value'] for r in spark.table('config').collect()}
RUN_START = datetime.now(timezone.utc)


## S3 SigV4 REST helpers (used only when `source_mode=s3_direct`)
Pure `requests` + stdlib — no boto3, no `%pip`. Mirrors `scripts/s3_rest.py`.


In [ ]:
import hashlib, hmac, os, time
import datetime as _dt
import urllib.parse as _url
import xml.etree.ElementTree as _ET
import requests

_EMPTY_SHA256 = hashlib.sha256(b"").hexdigest()

def _s3_sign(key, msg):
    return hmac.new(key, msg.encode("utf-8"), hashlib.sha256).digest()

def _s3_signing_key(secret, datestamp, region, service):
    k_date = _s3_sign(("AWS4" + secret).encode("utf-8"), datestamp)
    k_region = hmac.new(k_date, region.encode("utf-8"), hashlib.sha256).digest()
    k_service = hmac.new(k_region, service.encode("utf-8"), hashlib.sha256).digest()
    return hmac.new(k_service, b"aws4_request", hashlib.sha256).digest()

def _s3_canonical_query(params):
    if not params:
        return ""
    items = []
    for k in sorted(params):
        v = "" if params[k] is None else str(params[k])
        items.append(_url.quote(str(k), safe="") + "=" + _url.quote(v, safe=""))
    return "&".join(items)

def _s3_endpoint_parts(endpoint_url, bucket, key, addressing):
    p = _url.urlparse(endpoint_url)
    scheme = p.scheme or "https"
    ep_host = p.netloc
    enc_key = _url.quote(key, safe="/")
    if addressing == "virtual":
        host = bucket + "." + ep_host
        canonical_uri = "/" + enc_key
    else:
        host = ep_host
        canonical_uri = "/" + bucket + (("/" + enc_key) if key else "")
    return host, canonical_uri, scheme + "://" + host + canonical_uri

def _s3_signed_headers(method, host, canonical_uri, params, region, ak, sk, service="s3"):
    now = _dt.datetime.now(_dt.timezone.utc)
    amzdate = now.strftime("%Y%m%dT%H%M%SZ")
    datestamp = now.strftime("%Y%m%d")
    canonical_qs = _s3_canonical_query(params)
    canonical_headers = ("host:" + host + "\n"
                         "x-amz-content-sha256:" + _EMPTY_SHA256 + "\n"
                         "x-amz-date:" + amzdate + "\n")
    signed_headers = "host;x-amz-content-sha256;x-amz-date"
    canonical_request = "\n".join([method, canonical_uri, canonical_qs,
                                   canonical_headers, signed_headers, _EMPTY_SHA256])
    scope = datestamp + "/" + region + "/" + service + "/aws4_request"
    string_to_sign = "\n".join(["AWS4-HMAC-SHA256", amzdate, scope,
                                hashlib.sha256(canonical_request.encode("utf-8")).hexdigest()])
    signature = hmac.new(_s3_signing_key(sk, datestamp, region, service),
                         string_to_sign.encode("utf-8"), hashlib.sha256).hexdigest()
    authorization = ("AWS4-HMAC-SHA256 Credential=" + ak + "/" + scope +
                     ", SignedHeaders=" + signed_headers + ", Signature=" + signature)
    return {"Authorization": authorization, "x-amz-date": amzdate,
            "x-amz-content-sha256": _EMPTY_SHA256}

def _s3_signed_get(endpoint_url, bucket, key, region, ak, sk, params=None,
                   addressing="path", verify=True, stream=False, timeout=(10, 300)):
    host, canonical_uri, url = _s3_endpoint_parts(endpoint_url, bucket, key, addressing)
    headers = _s3_signed_headers("GET", host, canonical_uri, params, region, ak, sk)
    resp = requests.get(url, headers=headers, params=params, verify=verify,
                        stream=stream, timeout=timeout)
    if 300 <= resp.status_code < 400:
        raise requests.HTTPError("S3 " + str(resp.status_code) +
                                 " redirect (wrong region/endpoint?): " + resp.text[:300])
    return resp

def s3_list_objects(endpoint_url, bucket, region, ak, sk, prefix="",
                    addressing="path", verify=True, timeout=(10, 60)):
    """ListObjectsV2 across continuation tokens -> [{key,size,last_modified,etag}]."""
    ns = "{http://s3.amazonaws.com/doc/2006-03-01/}"
    out, token = [], None
    while True:
        params = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
        if token:
            params["continuation-token"] = token
        r = _s3_signed_get(endpoint_url, bucket, "", region, ak, sk,
                           params=params, addressing=addressing, verify=verify, timeout=timeout)
        r.raise_for_status()
        root = _ET.fromstring(r.content)
        for c in root.findall(ns + "Contents"):
            k = c.findtext(ns + "Key")
            if not k or k.endswith("/"):
                continue
            out.append({"key": k,
                        "size": int(c.findtext(ns + "Size") or 0),
                        "last_modified": c.findtext(ns + "LastModified"),
                        "etag": (c.findtext(ns + "ETag") or "").strip('"')})
        truncated = (root.findtext(ns + "IsTruncated") or "false").lower() == "true"
        token = root.findtext(ns + "NextContinuationToken")
        if not truncated or not token:
            break
    return out

def s3_get_bytes(endpoint_url, bucket, key, region, ak, sk,
                 addressing="path", verify=True, timeout=(10, 300)):
    r = _s3_signed_get(endpoint_url, bucket, key, region, ak, sk,
                       addressing=addressing, verify=verify, timeout=timeout)
    r.raise_for_status()
    return r.content


## Source-layer config
Resolves `source_mode` and (for `s3_direct`) the endpoint/bucket/region + S3 credentials from
Key Vault.


In [ ]:
# Source-layer config (mode-independent identity = src_key = object path relative to bucket root).
SOURCE_MODE   = cfg.get("source_mode", "s3_shortcut").strip().lower()   # s3_shortcut | s3_direct
SHORTCUT_ROOT = cfg.get("shortcut_root", "Files/s3_mmx_bucket").rstrip("/")
S3_ENDPOINT   = cfg.get("s3_endpoint_url", "").rstrip("/")
S3_BUCKET     = cfg.get("s3_bucket", "")
S3_PREFIX     = cfg.get("s3_prefix", "")
S3_REGION     = cfg.get("s3_region", "us-east-1")
S3_ADDRESSING = cfg.get("s3_addressing", "path").strip().lower()        # path | virtual
S3_VERIFY_TLS = cfg.get("s3_verify_tls", "true").strip().lower() != "false"

_S3_AK = _S3_SK = None
if SOURCE_MODE == "s3_direct":
    import notebookutils
    _VAULT = "https://" + cfg["kv_name"] + ".vault.azure.net/"
    _S3_AK = notebookutils.credentials.getSecret(_VAULT, cfg.get("s3_access_key_secret", "s3-access-key"))
    _S3_SK = notebookutils.credentials.getSecret(_VAULT, cfg.get("s3_secret_key_secret", "s3-secret-key"))
    if not (S3_ENDPOINT and S3_BUCKET):
        raise RuntimeError("source_mode=s3_direct requires s3_endpoint_url and s3_bucket in config")
    print("source: s3_direct ->", S3_ENDPOINT, "bucket=" + S3_BUCKET,
          "prefix=" + repr(S3_PREFIX), "region=" + S3_REGION, "addr=" + S3_ADDRESSING)
else:
    print("source: s3_shortcut ->", SHORTCUT_ROOT)


## Listing → common `scanned` DataFrame
Both modes produce the **same** schema keyed on `file_path` = src_key. Shortcut mode uses
`binaryFile` + `recursiveFileLookup` (column-pruned to path/length/mtime, so bytes are never
read); direct mode pages `ListObjectsV2`. `change_hash = sha2(src_key|size|mtime)` in both.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

if SOURCE_MODE == 's3_direct':
    objs = s3_list_objects(S3_ENDPOINT, S3_BUCKET, S3_REGION, _S3_AK, _S3_SK,
                           prefix=S3_PREFIX, addressing=S3_ADDRESSING, verify=S3_VERIFY_TLS)
    print('s3_direct objects listed:', len(objs))
    rows_schema = T.StructType([
        T.StructField('file_path', T.StringType()), T.StructField('file_size', T.LongType()),
        T.StructField('modified_datetime', T.StringType()), T.StructField('etag', T.StringType())])
    rows = [(o['key'], int(o['size']), o['last_modified'], o['etag']) for o in objs]
    base = spark.createDataFrame(rows, rows_schema) if rows else spark.createDataFrame([], rows_schema)
    scanned = (base
        .withColumn('file_name', F.element_at(F.split('file_path', '/'), -1))
        .withColumn('modified_datetime', F.to_timestamp('modified_datetime')))
else:
    marker = '/' + SHORTCUT_ROOT + '/'
    raw = (spark.read.format('binaryFile')
           .option('recursiveFileLookup', 'true')
           .load(SHORTCUT_ROOT)
           .select('path', 'length', 'modificationTime'))
    scanned = (raw
        # file_path = src_key: strip everything up to and including the shortcut root.
        .withColumn('file_path', F.expr("substring_index(path, '" + marker + "', -1)"))
        .withColumn('file_name', F.element_at(F.split('path', '/'), -1))
        .withColumn('file_size', F.col('length').cast('bigint'))
        .withColumn('modified_datetime', F.col('modificationTime'))
        .withColumn('etag', F.lit(None).cast('string')))

scanned = (scanned
    .withColumn('file_extension',
                F.lower(F.regexp_extract(F.col('file_name'), r'\.([^.]+)$', 1)))
    .withColumn('author', F.lit(None).cast('string'))
    .withColumn('content_hash', F.lit(None).cast('string'))
    .withColumn('change_hash',
                F.sha2(F.concat_ws('|', F.col('file_path'), F.col('file_size'),
                                   F.col('modified_datetime').cast('string')), 256))
    .withColumn('last_seen_utc', F.lit(RUN_START))
    .select('file_path','file_name','file_extension','file_size','modified_datetime',
            'author','etag','content_hash','change_hash','last_seen_utc'))

# Cache: this DataFrame feeds several downstream actions (count, the change MERGE, and the
# deletion sweep). Caching avoids recomputing the listing + transforms each time.
scanned = scanned.cache()

print('files discovered:', scanned.count())
scanned.select('file_path','file_size','change_hash').show(5, truncate=False)


## MERGE into `file_metadata` (state machine)
- new path → `new`
- existing path, `change_hash` changed → `reingest`
- existing path, unchanged → keep status, just refresh `last_seen_utc`


In [ ]:
from delta.tables import DeltaTable

now_str = RUN_START
tgt = DeltaTable.forName(spark, 'file_metadata')

(tgt.alias('t')
  .merge(scanned.alias('s'), 't.file_path = s.file_path')
  # changed content -> reingest
  .whenMatchedUpdate(
      condition='t.change_hash <> s.change_hash',
      set={
        'file_size': 's.file_size', 'modified_datetime': 's.modified_datetime',
        'file_name': 's.file_name', 'file_extension': 's.file_extension',
        'etag': 's.etag', 'change_hash': 's.change_hash',
        'last_seen_utc': 's.last_seen_utc',
        'process_status': F.lit('reingest'),
        'status_reason': F.lit(None).cast('string'),
        'status_updated_utc': F.lit(now_str),
      })
  # unchanged -> just mark as seen this run
  .whenMatchedUpdate(
      condition='t.change_hash = s.change_hash',
      set={'last_seen_utc': 's.last_seen_utc'})
  # brand new file
  .whenNotMatchedInsert(
      values={
        'file_path': 's.file_path', 'file_name': 's.file_name',
        'file_extension': 's.file_extension', 'file_size': 's.file_size',
        'modified_datetime': 's.modified_datetime', 'author': 's.author',
        'etag': 's.etag', 'content_hash': 's.content_hash',
        'change_hash': 's.change_hash', 'acl_version': F.lit(None).cast('string'),
        'last_seen_utc': 's.last_seen_utc',
        'process_status': F.lit('new'),
        'status_reason': F.lit(None).cast('string'),
        'retry_count': F.lit(0),
        'status_updated_utc': F.lit(now_str),
      })
  .execute())
print('merge complete')


## Deletion detection
Any row not seen in this scan (stale `last_seen_utc`) that isn't already `deleted` is flagged
`deleted`; nb_03 will purge its chunks from AI Search.


In [ ]:
(DeltaTable.forName(spark, 'file_metadata').alias('t')
  .merge(scanned.select('file_path').alias('s'), 't.file_path = s.file_path')
  .whenNotMatchedBySourceUpdate(
      condition="t.process_status <> 'deleted'",
      set={'process_status': F.lit('deleted'),
           'status_reason': F.lit('not_present_in_source'),
           'status_updated_utc': F.lit(now_str)})
  .execute())
print('deletion sweep complete')


## Summary


In [ ]:
(spark.table('file_metadata')
   .groupBy('process_status').count().orderBy('process_status').show())
